# Phase 8: PBR CO2 Fixation and Biomass Productivity

This notebook audits the locally supplied Zenodo supplementary workbook (DOI `10.5281/zenodo.22065573`; title identifies Chlorella sp.) before creating separate performance and environmental data layers. The PBR raw folder currently contains the Excel workbook and no separate PDF. It does not merge PBR data with Hiltunen and does not force predictive modeling when the independent sample structure is inadequate.

## 1. Hard dataset audit and reproducible pipeline

In [ ]:
from pathlib import Path
import importlib
import sys

SOURCE_PATH = Path("data/raw/pbr/Supplementary Data Excel.xlsx")
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((path for path in search_locations if (path / SOURCE_PATH).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("PBR workbook is missing under data/raw/pbr; place the supplied workbook there. No download was attempted.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.models.pbr_co2_analysis as pbr_analysis

pbr_analysis = importlib.reload(pbr_analysis)
analysis = pbr_analysis.run_pbr_analysis(PROJECT_ROOT)
inventory = analysis["inventory"]
performance = analysis["performance"]
environmental = analysis["environmental"]
target_summary = analysis["target_summary"]
environmental_features = analysis["environmental_features"]
model_results = analysis["model_results"]
print("Source workbook:", SOURCE_PATH)
print("Performance rows/columns:", performance.shape)
print("Environmental observations (long):", environmental.shape)
print("Raw workbook remains read-only; no Hiltunen data was loaded or joined.")

## 2. Workbook sheet inventory

In [ ]:
display(inventory[["Source_Sheet", "Rows", "Columns_Count", "Nonempty_Rows", "Duplicate_Rows_As_Read", "Column_Names", "Missing_Counts_By_Column"]])
print("Period Performance is the primary performance table; environmental series remain separate.")

## 4. Separate performance and environmental layers

The performance table remains 12 rows. High-frequency environment readings are normalized as separate timestamp-variable observations; monthly meteorological day values are included separately. `Raw_Value` preserves textual marks such as `T` and `-`; no values are imputed.

Period is a descriptive linkage key because period sensor sheets align with the performance period start dates. However, complete sensor windows overlap the period outcomes, and only three independent period windows exist. They are therefore not leakage-safe predictors and are not used for performance correlations or regression. The full-window period summaries below are descriptive only.

In [ ]:
display(performance)
display(target_summary)
print("Unique periods:", sorted(performance["Period"].unique().tolist()))
print("PBR types:", sorted(performance["PBR type"].unique().tolist()))
print("Cultivation conditions:", sorted(performance["Cultivation condition"].unique().tolist()))
print("Organic-carbon replicate audit:", analysis["carbon_validation"])
print("CO2 fixation formula audit:", analysis["formula_check"])

## 4. Separate performance and environmental layers
The performance table remains 12 rows. High-frequency environment readings are normalized as separate timestamp-variable observations; monthly meteorological day values are included separately. `Raw_Value` preserves textual marks such as `T` and `-`; no values are imputed.

In [ ]:
display(environmental.groupby(["Observation_Source", "Variable", "Unit"], dropna=False).agg(
    Observation_Count=("Source_Row_1Based", "size"),
    Numeric_Value_Count=("Value", "count"),
    Timestamp_Count=("Timestamp", "count"),
).reset_index())
display(environmental_features)
print("Period is an explicit descriptive key: performance period start dates align with the corresponding period sensor windows.")
print("The full sensor-window summaries include measurements during the performance period, so they are descriptive and not pre-outcome forecast features.")

## 5. Descriptive PBR type and cultivation-condition comparisons
Each PBR type and each condition has six performance rows (three periods times the other two-level factor). These are small repeated period samples; observed differences are not interpreted causally.

In [ ]:
print("By PBR type:")
display(analysis["by_type"])
print("By cultivation condition:")
display(analysis["by_condition"])

from IPython.display import Image, display
for filename in [
    "pbr_co2_fixation_by_type.png",
    "pbr_biomass_productivity_by_type.png",
    "pbr_growth_rate_by_type.png",
]:
    print(filename)
    display(Image(filename=str(analysis["figure_directory"] / filename)))

## 6. Environmental observations over time
Plots show supplied measurements at their recorded timestamps. No interpolation, smoothing, or outcome join is performed.

In [ ]:
for filename in ["pbr_temperature_over_time.png", "pbr_solar_radiation_over_time.png"]:
    print(filename)
    display(Image(filename=str(analysis["figure_directory"] / filename)))

## 7. Predictive modeling gate and leakage assessment
The workbook offers 12 performance records grouped in only three periods and no performance replicate/run ID. Environmental measurements cover the same period windows as the period outcomes, without a documented within-period outcome timestamp. Aggregating the complete window would expose contemporaneous/post-outcome information. Therefore, no Linear Regression or Random Forest model, split, or metric was produced. CO2 fixation is also exactly derived from biomass productivity and carbon fraction, so those fields cannot independently predict it.

In [ ]:
display(model_results)

## 8. Separation from Hiltunen and scientific interpretation
This is a separate PBR performance component for Chlorella sp. AARL G049. The Hiltunen dataset covers experimental Chlorella growth/biochemistry and is not joined here. Future transfer would require alignment of species/strain, units, measurement protocols, features, and independent validation.

The current workbook supports descriptive PBR-period summaries and characterization of environmental sensor records. It does not support a reliable independent CO2-fixation prediction or autonomous control claim. A future dataset should include more independent cultivation runs, replicate identifiers, outcome timing, and environmental windows recorded strictly before outcomes. No ANN/LSTM/GRU is justified by this dataset; collect more independent PBR data first.